# AntimEnv GRPO Training
## Training Llama-3.2-1B to coordinate post-death workflows in India

## Cell 1: Install Dependencies
Install all required packages for GRPO training with TRL, Unsloth, and AntimEnv

In [ ]:
!pip install -q -U "trl==1.2.0" "transformers>=5.2.0" "openenv-core>=0.2.3" "unsloth" wandb datasets

## Cell 2: Authenticate with Hugging Face and Weights & Biases
Set up authentication tokens for model hub access and experiment tracking

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

import wandb
wandb.init(project="antim-env-grpo", name="llama3.2-1b-run-1")
print("✓ Authentication complete")

## Cell 3: Load Llama-3.2-1B with Unsloth
Load the base model with 4-bit quantization and LoRA adapters for efficient training

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=4096,
    load_in_4bit=True,
    fast_inference=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

print("✓ Model loaded successfully")
print(f"Model: Llama-3.2-1B with LoRA (r=16)")
print(f"Max sequence length: 4096")

## Cell 4: AntimEnv Wrapper for TRL
Create a wrapper class that connects to the AntimEnv HF Space and provides the interface for GRPO training

In [ ]:
"""
In-process AntimEnvWrapper.

We import AntimEnvironment directly instead of going through HTTP. This is
- deterministic (no network jitter)
- 50–100x faster per step
- doesn't depend on the HF Space being live during training

The HF Space deployment is a *separate* artifact for judges to verify
post-submission. Training runs locally (or on the Colab VM) against the
in-process env.
"""
from antim_env import AntimEnvironment, AntimAction


class AntimEnvWrapper:
    """Thin wrapper that exposes the in-process AntimEnvironment as something
    a reward function can roll out one tool call at a time."""

    def __init__(self, case_id: str | None = None):
        self.env = AntimEnvironment()
        self.case_id = case_id
        self.last_reward = 0.0
        self.done = False
        self.reset()

    def reset(self) -> str:
        obs = self.env.reset(case_id=self.case_id)
        self.last_reward = 0.0
        self.done = False
        return obs.message

    def call_tool(self, tool_name: str, **params) -> tuple[str, float, bool]:
        """Execute a single tool call. Returns (message, reward, done)."""
        action = AntimAction(tool_name=tool_name, parameters=params or {})
        obs = self.env.step(action)
        self.last_reward = float(obs.reward)
        self.done = bool(obs.done)
        return obs.message, self.last_reward, self.done


print("✓ AntimEnvWrapper (in-process) ready")

## Cell 5: Create Training Dataset
Build a dataset of Indian family cases for GRPO training

In [ ]:
"""
Training dataset.

Each row is a *prompt* whose completion the model is supposed to produce as a
JSON array of tool calls. We attach `case_id` to every row so the reward
function can roll out the model's plan against the matching env instance.

Train cases: CASE_001..CASE_008 (8). Held-out eval cases: CASE_009, CASE_010 (2).
"""
from datasets import Dataset

SYSTEM_PROMPT = """You are an AI coordinator helping an Indian family after a death.
Output a JSON array of tool calls in optimal order. Stop when all critical tasks are done.

Output format (strict):
[
  {"tool": "<name>", "parameters": {...}},
  ...
]

Tools (exact names + parameters):
- book_funeral_service(vendor_id: str, slot_time: str)
- submit_death_certificate_application(municipality_id: str)
- advance_time(days: int)                          # 1..7
- notify_bank(bank_id: str, account_type: str)     # account_type: "savings"|"fd"|"locker"|"loan"
- file_insurance_claim(policy_id: str, claim_type: str)
- check_government_scheme_eligibility(scheme_name: str)
- escalate_delay(office_id: str, reason: str)
- get_case_context()
- check_document_status(document_type: str)
- get_next_critical_deadline()

Hard constraints (violations are penalized):
- Death certificate requires the funeral to be completed first.
- Bank notification requires the death certificate to be obtained first.
- Insurance claim requires the death certificate to be obtained first.
- The 21-day legal deadline for the death certificate is asymmetrically penalized.

Output ONLY the JSON array. No prose.
"""

TRAIN_CASES = [f"CASE_00{i}" for i in range(1, 9)]   # CASE_001..CASE_008
EVAL_CASES  = ["CASE_009", "CASE_010"]


def _make_row(case_id: str) -> dict:
    env = AntimEnvWrapper(case_id=case_id)
    intro = env.reset()
    return {
        "case_id": case_id,
        "prompt": (
            f"{SYSTEM_PROMPT}\n\n"
            f"Case briefing:\n{intro}\n\n"
            f"Your plan as a JSON array:\n"
        ),
    }


# 8 train cases × 25 repeats = 200 prompts (random sampling stays diverse).
train_rows = [_make_row(c) for c in TRAIN_CASES * 25]
eval_rows  = [_make_row(c) for c in EVAL_CASES * 5]

dataset      = Dataset.from_list(train_rows)
eval_dataset = Dataset.from_list(eval_rows)

print(f"✓ Train dataset: {len(dataset)} rows ({len(TRAIN_CASES)} cases × 25)")
print(f"✓ Eval dataset:  {len(eval_dataset)} rows ({len(EVAL_CASES)} held-out cases × 5)")
print("\nSample prompt (first 240 chars):")
print(dataset[0]["prompt"][:240] + "...")

## Cell 6: Define Reward Function
Create the reward function that extracts rewards from the environment

In [ ]:
"""
Rollout-based reward function.

GRPO calls this with `prompts` (the inputs) and `completions` (what the model
generated). For each completion we:
  1. parse the JSON array of tool calls,
  2. spin up a fresh env for the row's case_id,
  3. step through the actions one by one,
  4. accumulate the env's per-step reward as the trajectory return.

Malformed completions are penalized so the model is pressured into emitting
valid JSON. Each tool call is capped at 20 to bound rollout cost.
"""
import json
import re
from typing import Any

# Fallback: extract the first top-level JSON array embedded in prose.
_JSON_ARRAY_RE = re.compile(r"\[\s*(?:\{.*?\}\s*,?\s*)*\]", re.DOTALL)

INVALID_FORMAT_PENALTY = -0.5
INVALID_CALL_PENALTY   = -0.05
MAX_TRAJECTORY_LEN     = 20


def parse_actions(completion: str) -> list[dict[str, Any]] | None:
    """Extract a JSON array of {tool, parameters} dicts from a completion.

    Tries strict whole-string parsing first (handles []), then falls back to
    regex-extracting the first JSON array embedded in prose.
    """
    if not isinstance(completion, str):
        return None
    stripped = completion.strip()
    # Strict path: whole completion is a JSON array (also handles []).
    if stripped.startswith("[") and stripped.endswith("]"):
        try:
            parsed = json.loads(stripped)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
    # Fallback: regex-extract first array from arbitrary prose.
    m = _JSON_ARRAY_RE.search(completion)
    if not m:
        return None
    try:
        parsed = json.loads(m.group(0))
    except Exception:
        return None
    return parsed if isinstance(parsed, list) else None


def antim_reward_func(prompts, completions, **kwargs) -> list[float]:
    """TRL-compatible reward function.

    prompts:     list[str]
    completions: list[str]
    kwargs:      includes 'case_id' (list[str]), passed via dataset columns.

    Returns: list[float] — one cumulative reward per completion.
    """
    case_ids = kwargs.get("case_id") or [None] * len(completions)
    rewards: list[float] = []

    for completion, case_id in zip(completions, case_ids):
        actions = parse_actions(completion)
        if actions is None:
            rewards.append(INVALID_FORMAT_PENALTY)
            continue

        env = AntimEnvWrapper(case_id=case_id)
        env.reset()
        cumulative = 0.0

        for act in actions[:MAX_TRAJECTORY_LEN]:
            if not isinstance(act, dict):
                cumulative += INVALID_CALL_PENALTY
                continue
            tool = act.get("tool")
            params = act.get("parameters", {}) or {}
            if not isinstance(tool, str) or not isinstance(params, dict):
                cumulative += INVALID_CALL_PENALTY
                continue
            try:
                _msg, step_reward, done = env.call_tool(tool, **params)
            except Exception:
                cumulative += INVALID_CALL_PENALTY
                continue
            cumulative += step_reward
            if done:
                break

        rewards.append(float(cumulative))

    return rewards


print("✓ antim_reward_func ready (rolls out trajectory inside reward fn)")

## Cell 7: Configure GRPO Training
Set up the GRPO trainer with model, dataset, and training parameters

In [ ]:
"""
GRPO training configuration.

We DROP `environment_factory`: rollouts now happen inside antim_reward_func,
so TRL doesn't need to know about the env. This is the simpler path — we
trade TRL's multi-turn machinery for a straightforward single-shot
"emit-a-plan, score-the-plan" loop, which is easier to debug and converges
fine for trajectories <= 20 tool calls.
"""
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    use_vllm=True,
    vllm_mode="colocate",
    num_generations=4,
    gradient_accumulation_steps=8,
    max_completion_length=1024,
    report_to="wandb",
    log_completions=True,
    run_name="antim-llama32-1b-grpo",
    num_train_epochs=2,
    logging_steps=5,
    save_steps=100,
    push_to_hub=False,
    output_dir="./antim-grpo-output",
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    reward_funcs=[antim_reward_func],
    args=training_args,
)

print("✓ GRPO trainer configured")
print(f"   epochs={training_args.num_train_epochs}, "
      f"generations/prompt={training_args.num_generations}, "
      f"max_completion_length={training_args.max_completion_length}")

## Cell 8: Run GRPO Training
Start the training loop and monitor progress on Weights & Biases

In [ ]:
print("🚀 Starting GRPO training...")
print("📊 Watch reward curve at: https://wandb.ai")
print("⏱️  This will take approximately 30-60 minutes on a single GPU\n")

try:
    trainer.train()
    print("\n✓ Training complete!")
except Exception as e:
    print(f"\n⚠️  Training interrupted: {e}")
    print("Saving checkpoint...")

## Cell 9: Save Trained Model
Save the fine-tuned model and tokenizer locally

In [ ]:
model.save_pretrained("antim-llama3.2-1b-lora")
tokenizer.save_pretrained("antim-llama3.2-1b-lora")

print("✓ Model saved locally to: antim-llama3.2-1b-lora")
print("\nTo push to Hugging Face Hub:")
print("  model.push_to_hub('your-username/antim-llama3.2-1b-lora')")
print("  tokenizer.push_to_hub('your-username/antim-llama3.2-1b-lora')")

## Cell 10: Evaluate Base vs Trained Model
**The Money Shot**: Compare base model vs trained model on CASE_001

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def run_episode(model, tokenizer, case_id: str = "CASE_001", max_steps: int = 5) -> float:
    """
    Run a single episode and return the final reward.

    Args:
        model: The language model
        tokenizer: The tokenizer
        case_id: Case ID to run
        max_steps: Maximum steps in the episode

    Returns:
        Final reward from the episode
    """
    env = AntimEnvWrapper(case_id=case_id)
    initial_prompt = env.reset()

    # Format prompt for the model
    system_prompt = """You are a coordination agent helping an Indian family navigate post-death administrative tasks. Use the available tools to complete critical tasks in the correct order."""
    full_prompt = f"{system_prompt}\n\n{initial_prompt}\n\nNext action:"

    total_reward = 0.0

    for step in range(max_steps):
        # Generate action
        inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.7,
                top_p=0.9,
            )
        action_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        # Parse action (simplified - in practice would need better parsing)
        if "get_case_context" in action_text.lower():
            response = env.get_case_context()
        elif "check_document" in action_text.lower():
            response = env.check_document_status("death_certificate")
        elif "book_funeral" in action_text.lower():
            response = env.book_funeral_service("vendor_001", "10:00 AM")
        elif "death_certificate" in action_text.lower():
            response = env.submit_death_certificate_application("municipality_001")
        elif "notify_bank" in action_text.lower():
            response = env.notify_bank("SBI", "savings")
        elif "insurance" in action_text.lower():
            response = env.file_insurance_claim("LIC-001", "death")
        elif "scheme" in action_text.lower():
            response = env.check_government_scheme_eligibility("widow_pension")
        elif "deadline" in action_text.lower():
            response = env.get_next_critical_deadline()
        elif "advance" in action_text.lower():
            response = env.advance_time(3)
        else:
            response = env.get_case_context()

        total_reward += env.last_reward
        full_prompt += f"\n{action_text}\n\nResponse: {response[:200]}...\n\nNext action:"

    return total_reward


print("🧪 Evaluating models on CASE_001...\n")

# Load base model
print("Loading base model (Llama-3.2-1B)...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

print("Running base model episode...")
base_reward = run_episode(base_model, base_tokenizer, case_id="CASE_001", max_steps=5)
print(f"Base model final reward: {base_reward:.4f}\n")

# Load trained model
print("Loading trained model...")
trained_model = FastLanguageModel.from_pretrained(
    model_name="antim-llama3.2-1b-lora",
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(trained_model)

print("Running trained model episode...")
trained_reward = run_episode(trained_model, tokenizer, case_id="CASE_001", max_steps=5)
print(f"Trained model final reward: {trained_reward:.4f}\n")

# Calculate improvement
if base_reward > 0:
    improvement = ((trained_reward - base_reward) / base_reward) * 100
else:
    improvement = (trained_reward - base_reward) * 100

print("="*60)
print("📊 EVALUATION RESULTS")
print("="*60)
print(f"Base model reward:    {base_reward:.4f}")
print(f"Trained model reward: {trained_reward:.4f}")
print(f"Improvement:          {improvement:+.1f}%")
print("="*60)

if trained_reward > base_reward:
    print("\n✅ SUCCESS: Trained model outperforms base model!")
else:
    print("\n⚠️  Trained model needs more training or tuning.")